# 07 - Limpieza bibliográfica autónoma

Esta libreta usa **únicamente** `autores_unam_normalizados.csv`.

No depende del archivo maestro ni de otros canónicos.

Reglas principales:
- conserva exactamente las 16 columnas canónicas;
- no modifica autores, afiliaciones, Area ni SubArea;
- no deduplica publicaciones;
- limpia Titulo, Año, ISBN, ISSN, Doi, URL, Keywords y Abstract;
- nunca reconstruye un ISBN desde notación científica;
- si un ISBN científico no puede recuperarse dentro del propio archivo, queda vacío y se registra para revisión.


In [1]:
import os
import re
import html
import unicodedata
import pandas as pd

archivo_entrada = "../04_Limpieza/02_normalizacion/autores_unam_normalizados.csv"

carpeta_salida = "../04_Limpieza/03_limpieza_bibliografica"
archivo_salida = f"{carpeta_salida}/autores_unam_limpios.csv"
archivo_revision = f"{carpeta_salida}/casos_revision_bibliografica.csv"

os.makedirs(carpeta_salida, exist_ok=True)

columnas = [
    "Base_origen", "Fuente_origen", "indice", "Titulo", "Año",
    "Autor_norm", "Afiliacion1", "Afiliacion2", "ISBN", "ISSN",
    "Doi", "URL", "Area", "SubArea", "Keywords", "Abstract"
]

clave_publicacion = ["Base_origen", "indice"]


In [2]:
# Funciones de limpieza

VACIOS = {
    "", "nan", "none", "null", "n/a", "na", "nd", "n/d",
    "-", "--", "—", "sin información", "sin informacion", "no disponible"
}


def nfc(texto):
    return unicodedata.normalize("NFC", texto or "")


def decodificar_html(texto):
    texto = str(texto or "")
    for _ in range(4):
        nuevo = html.unescape(texto)
        if nuevo == texto:
            break
        texto = nuevo
    return nfc(texto)


def limpiar_espacios(texto):
    texto = decodificar_html(texto)
    texto = texto.replace("\u00a0", " ")
    texto = texto.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    return re.sub(r"\s+", " ", texto).strip()


def limpiar_vacio(texto):
    texto = limpiar_espacios(texto)
    return "" if texto.casefold() in VACIOS else texto


def quitar_html(texto):
    texto = decodificar_html(texto)
    texto = re.sub(r"(?i)<\s*br\s*/?\s*>", " ", texto)
    texto = re.sub(r"(?i)</?\s*(?:p|div|li)\b[^>]*>", " ", texto)
    texto = re.sub(r"<[^>]+>", "", texto)
    return limpiar_espacios(texto)


def limpiar_titulo(texto):
    return quitar_html(limpiar_vacio(texto))


def limpiar_anio(texto):
    texto = limpiar_vacio(texto)
    if not texto:
        return ""

    m = re.fullmatch(r"(\d{4})(?:\.0+)?", texto)
    if m:
        return m.group(1)

    años = re.findall(r"\b(?:19|20)\d{2}\b", texto)
    if len(años) == 1:
        return años[0]

    return ""


def isbn13_valido(isbn):
    if not re.fullmatch(r"\d{13}", isbn):
        return False
    total = sum((1 if i % 2 == 0 else 3) * int(c) for i, c in enumerate(isbn[:12]))
    return (10 - total % 10) % 10 == int(isbn[-1])


def isbn10_valido(isbn):
    isbn = isbn.upper()
    if not re.fullmatch(r"\d{9}[\dX]", isbn):
        return False
    total = 0
    for i, c in enumerate(isbn):
        valor = 10 if c == "X" else int(c)
        total += (10 - i) * valor
    return total % 11 == 0


def limpiar_isbn(texto):
    texto = limpiar_vacio(texto)
    if not texto:
        return ""

    # Una notación científica ya perdió información de dígitos.
    # No se reconstruye ni se redondea.
    if re.search(r"(?i)\d(?:\.\d+)?e[+-]?\d+", texto):
        return ""

    resultado = []

    # ISBN-13 con o sin guiones/espacios.
    for m in re.finditer(r"(?<!\d)(97[89](?:[-\s]?\d){10})(?!\d)", texto):
        isbn = re.sub(r"\D", "", m.group(1))
        if isbn13_valido(isbn) and isbn not in resultado:
            resultado.append(isbn)

    # ISBN-10.
    for m in re.finditer(
        r"(?<![0-9A-Za-z])(\d(?:[-\s]?\d){8}[-\s]?[0-9Xx])(?![0-9A-Za-z])",
        texto
    ):
        isbn = re.sub(r"[^0-9Xx]", "", m.group(1)).upper()
        if isbn10_valido(isbn) and isbn not in resultado:
            resultado.append(isbn)

    # Caso simple: el campo completo es el ISBN.
    if not resultado:
        limpio = re.sub(r"(?i)\bISBN(?:-1[03])?\b", "", texto)
        limpio = re.sub(r"[^0-9Xx]", "", limpio).upper()
        if isbn13_valido(limpio) or isbn10_valido(limpio):
            resultado.append(limpio)

    return "; ".join(resultado)


def issn_valido(issn):
    limpio = issn.replace("-", "").upper()
    if not re.fullmatch(r"\d{7}[\dX]", limpio):
        return False

    total = sum((8 - i) * int(limpio[i]) for i in range(7))
    control = (11 - total % 11) % 11
    esperado = "X" if control == 10 else str(control)
    return limpio[-1] == esperado


def limpiar_issn(texto):
    texto = limpiar_vacio(texto)
    if not texto:
        return ""

    resultado = []

    for parte in re.split(r"[;|,]", texto):
        parte = re.sub(r"(?i)\b(?:e-?ISSN|ISSN)\b[:\s]*", "", parte)
        limpio = re.sub(r"[^0-9Xx]", "", parte).upper()

        candidatos = []

        if len(limpio) == 8:
            candidatos.append(limpio)

        # Excel puede haber eliminado ceros iniciales.
        # Solo se restauran si el checksum resultante es válido.
        elif 1 <= len(limpio) < 8 and limpio.isdigit():
            candidatos.append(limpio.zfill(8))

        for candidato in candidatos:
            issn = candidato[:4] + "-" + candidato[4:]
            if issn_valido(issn) and issn not in resultado:
                resultado.append(issn)

    return "; ".join(resultado)


DOI_RE = re.compile(r"10\.\d{4,9}/[^\s\"'<>]+", re.I)


def limpiar_doi(texto):
    texto = limpiar_vacio(texto)
    if not texto:
        return ""

    texto = decodificar_html(texto)
    texto = re.sub(
        r"(?i)^\s*(?:https?://(?:dx\.)?doi\.org/|(?:dx\.)?doi\.org/|doi\s*:\s*)",
        "",
        texto
    ).strip()

    m = DOI_RE.search(texto)
    if not m:
        return ""

    return m.group(0).strip().rstrip(".,;").lower()


def limpiar_url(texto):
    texto = limpiar_vacio(texto)
    if not texto:
        return ""

    texto = decodificar_html(texto).strip()
    if re.fullmatch(r"(?i)https?://\S+", texto):
        return texto

    return ""


URL_RE = re.compile(r"(?i)https?://[^;\s]+|www\.[^;\s]+")
PARAM_RE = re.compile(
    r"(?i)\b(?:arnumber|isnumber|querytext|refinements?|tag|searchwithin|highlight)\s*="
)


def limpiar_keywords(texto):
    texto = limpiar_vacio(texto)
    if not texto:
        return ""

    texto = quitar_html(texto)
    texto = URL_RE.sub(" ", texto)

    resultado = []
    vistos = set()

    for parte in re.split(r"[;|]", texto):
        parte = limpiar_espacios(parte).strip(" ,")

        if not parte:
            continue

        if PARAM_RE.search(parte):
            continue

        if ("=" in parte and "&" in parte) or parte.startswith("?"):
            continue

        clave = parte.casefold()
        if clave not in vistos:
            vistos.add(clave)
            resultado.append(parte)

    return "; ".join(resultado)


def limpiar_abstract(texto):
    texto = limpiar_vacio(texto)
    if not texto:
        return ""
    return quitar_html(texto)


In [3]:
# Leer el único archivo de entrada
entrada_fisica = pd.read_csv(
    archivo_entrada,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig"
)

print("Forma física de entrada:", entrada_fisica.shape)

faltantes = [c for c in columnas if c not in entrada_fisica.columns]
if faltantes:
    raise ValueError(f"Faltan columnas canónicas: {faltantes}")

# Elimina automáticamente columnas extra como Unnamed.
entrada = entrada_fisica[columnas].copy()

salida = entrada.copy()

funciones = {
    "Titulo": limpiar_titulo,
    "Año": limpiar_anio,
    "ISBN": limpiar_isbn,
    "ISSN": limpiar_issn,
    "Doi": limpiar_doi,
    "URL": limpiar_url,
    "Keywords": limpiar_keywords,
    "Abstract": limpiar_abstract,
}

for campo, funcion in funciones.items():
    salida[campo] = salida[campo].map(funcion)

print("Filas:", len(salida))
print("Columnas canónicas:", len(salida.columns))


Forma física de entrada: (4266, 20)
Filas: 4266
Columnas canónicas: 16


In [4]:
# Propagar una única versión bibliográfica dentro de cada publicación.
# Como la entrada tiene una fila por autor, Base_origen + indice puede repetirse.

campos_publicacion = [
    "Titulo", "Año", "ISBN", "ISSN", "Doi", "URL",
    "Area", "SubArea", "Keywords", "Abstract"
]

for campo in campos_publicacion:
    for _, indices in salida.groupby(clave_publicacion, sort=False).groups.items():
        indices = list(indices)
        valores = [salida.at[i, campo] for i in indices]
        valores_no_vacios = [v for v in valores if str(v).strip()]

        if not valores_no_vacios:
            valor_final = ""
        else:
            conteos = pd.Series(valores_no_vacios).value_counts()
            valor_final = conteos.index[0]

        salida.loc[indices, campo] = valor_final


In [5]:
# Crear lista de casos que no pueden recuperarse usando solamente este archivo.
casos = []

for (base, indice), grupo_antes in entrada.groupby(clave_publicacion, sort=False):
    grupo_despues = salida[
        (salida["Base_origen"] == base) & (salida["indice"] == indice)
    ]

    titulo = grupo_antes["Titulo"].iloc[0]

    revisiones = [
        ("Año", "ANIO_NO_RECUPERABLE"),
        ("ISBN", "ISBN_NO_RECUPERABLE"),
        ("ISSN", "ISSN_INVALIDO"),
        ("Doi", "DOI_INVALIDO"),
        ("URL", "URL_INVALIDA"),
    ]

    for campo, problema in revisiones:
        antes = grupo_antes[campo].iloc[0].strip()
        despues = grupo_despues[campo].iloc[0].strip()

        if antes and not despues:
            casos.append({
                "Base_origen": base,
                "indice": indice,
                "Titulo": titulo,
                "Campo": campo,
                "Valor_original": antes,
                "Problema": problema,
                "Accion_recomendada": "Revisar manualmente al final; no inventar el valor."
            })

casos_revision = pd.DataFrame(casos)

print("Casos de revisión bibliográfica:", len(casos_revision))
if len(casos_revision):
    print(casos_revision["Problema"].value_counts())


Casos de revisión bibliográfica: 63
Problema
ISBN_NO_RECUPERABLE    63
Name: count, dtype: int64


In [6]:
# Validaciones finales
assert len(salida) == len(entrada), "Cambió el número de filas."
assert list(salida.columns) == columnas, "La salida no tiene exactamente 16 columnas."

# Estas columnas no pueden modificarse en esta fase.
for campo in [
    "Base_origen", "Fuente_origen", "indice",
    "Autor_norm", "Afiliacion1", "Afiliacion2", "Area", "SubArea"
]:
    assert entrada[campo].equals(salida[campo]), f"Se modificó {campo}."

assert salida["SubArea"].str.strip().eq("").all(), "SubArea dejó de estar vacía."

# ISBN
isbn_cientifico = salida["ISBN"].str.contains(r"(?i)e[+-]?\d+", regex=True).sum()
assert isbn_cientifico == 0, f"Quedan {isbn_cientifico} ISBN en notación científica."

for valor in salida["ISBN"]:
    if not valor:
        continue
    for isbn in valor.split("; "):
        assert isbn13_valido(isbn) or isbn10_valido(isbn), f"ISBN inválido: {isbn}"

# ISSN
for valor in salida["ISSN"]:
    if not valor:
        continue
    for issn in valor.split("; "):
        assert re.fullmatch(r"\d{4}-[\dX]{4}", issn), f"ISSN mal formado: {issn}"
        assert issn_valido(issn), f"ISSN inválido: {issn}"

# Año, DOI y Keywords
assert salida["Año"].map(
    lambda x: x == "" or bool(re.fullmatch(r"\d{4}", x))
).all(), "Hay años mal formados."

assert salida["Doi"].map(
    lambda x: x == "" or (x.startswith("10.") and " " not in x)
).all(), "Hay DOI mal formados."

assert not salida["Doi"].str.contains(r"(?i)doi\.org", regex=True).any()

assert not salida["Keywords"].str.contains(
    r"(?i)https?://|www\.|arnumber=|isnumber=|querytext=",
    regex=True
).any(), "Queda basura web dentro de Keywords."

# Todos los autores de la misma publicación comparten bibliografía.
for campo in campos_publicacion:
    conflictos = salida.groupby(clave_publicacion)[campo].nunique(dropna=False).gt(1).sum()
    assert conflictos == 0, f"Hay {conflictos} publicaciones con conflicto en {campo}."

print("VALIDACIONES COMPLETADAS CORRECTAMENTE")
print("Filas finales:", len(salida))
print("Columnas finales:", len(salida.columns))
print("ISBN en notación científica:", isbn_cientifico)
print("ISBN no vacíos:", salida["ISBN"].str.strip().ne("").sum())
print("ISSN no vacíos:", salida["ISSN"].str.strip().ne("").sum())


VALIDACIONES COMPLETADAS CORRECTAMENTE
Filas finales: 4266
Columnas finales: 16
ISBN en notación científica: 0
ISBN no vacíos: 1847
ISSN no vacíos: 3276


In [7]:
# Guardar resultados
salida.to_csv(
    archivo_salida,
    index=False,
    encoding="utf-8-sig"
)

if len(casos_revision):
    casos_revision.to_csv(
        archivo_revision,
        index=False,
        encoding="utf-8-sig"
    )
else:
    pd.DataFrame(columns=[
        "Base_origen", "indice", "Titulo", "Campo", "Valor_original",
        "Problema", "Accion_recomendada"
    ]).to_csv(archivo_revision, index=False, encoding="utf-8-sig")

print("Archivo limpio:", archivo_salida)
print("Casos de revisión:", archivo_revision)


Archivo limpio: ../04_Limpieza/03_limpieza_bibliografica/autores_unam_limpios.csv
Casos de revisión: ../04_Limpieza/03_limpieza_bibliografica/casos_revision_bibliografica.csv
